# Multiclass Classic ML Pipeline for CSI Dataset

This notebook builds a multiclass pipeline for `wifi_data_set_fixed`:
1. Read every `.data` file (each file is one data unit).
2. Build a 1D amplitude-over-time signal per unit and apply median filter with `median_width`.
3. Scale every unit using global min and max amplitude from train split only.
4. Create an embedding with statistical and spectral features.
5. Train multiclass models with labels: label_00 -> 0, label_01 -> 1, label_02 -> 2, label_03 -> 3.

In [ ]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

# Make project root importable so `tools` can be imported from notebooks/
project_root = Path.cwd()
if not (project_root / 'tools').exists() and (project_root.parent / 'tools').exists():
    project_root = project_root.parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from tools.csi_parser import Parser
from tools.filters import median_filter

# Main parameters
dataset_root = project_root / 'wifi_data_set_fixed'
median_width = 5  # must be odd

if median_width % 2 == 0:
    raise ValueError('median_width must be odd')

np.random.seed(42)

label_to_class = {
    'label_00': 0,
    'label_01': 1,
    'label_02': 2,
    'label_03': 3,
}
class_to_label = {v: k for k, v in label_to_class.items()}

In [ ]:
def parse_path_meta(file_path: Path) -> dict:
    parts = file_path.parts
    return {
        'person_id': next((p for p in parts if p.startswith('id_person_')), 'unknown'),
        'label': next((p for p in parts if p.startswith('label_')), 'unknown'),
        'test_id': next((p for p in parts if p.startswith('test_')), 'unknown'),
    }


def skewness(x: np.ndarray) -> float:
    x = np.asarray(x, dtype=np.float64)
    mu = x.mean()
    sigma = x.std()
    if sigma == 0:
        return 0.0
    z = (x - mu) / sigma
    return float(np.mean(z ** 3))


def kurtosis_excess(x: np.ndarray) -> float:
    x = np.asarray(x, dtype=np.float64)
    mu = x.mean()
    sigma = x.std()
    if sigma == 0:
        return 0.0
    z = (x - mu) / sigma
    return float(np.mean(z ** 4) - 3.0)


def spectral_features(signal: np.ndarray) -> tuple[float, float, float, float, float]:
    x = np.asarray(signal, dtype=np.float64)
    n = len(x)
    if n < 2:
        return 0.0, 0.0, 0.0, 0.0, 0.0

    spectrum = np.fft.rfft(x)
    power = np.abs(spectrum) ** 2
    freqs = np.fft.rfftfreq(n, d=1.0)

    power_sum = power.sum()
    if power_sum <= 0:
        return 0.0, 0.0, 0.0, 0.0, 0.0

    centroid = float((freqs * power).sum() / power_sum)
    spread = float(np.sqrt(((freqs - centroid) ** 2 * power).sum() / power_sum))

    p_norm = power / power_sum
    eps = 1e-12
    entropy = float(-(p_norm * np.log2(p_norm + eps)).sum())

    dominant_idx = int(np.argmax(power))
    dominant_freq = float(freqs[dominant_idx])
    rolloff_threshold = 0.85 * power_sum
    rolloff = float(freqs[np.searchsorted(np.cumsum(power), rolloff_threshold)])

    return centroid, spread, entropy, dominant_freq, rolloff


def build_unit_signal(df: pd.DataFrame, window: int) -> np.ndarray:
    # For each packet, reduce subcarrier amplitudes to one value by averaging.
    amp_matrix = np.stack(df['amplitude'].to_numpy(), axis=0).astype(np.float64)
    raw_time_signal = amp_matrix.mean(axis=1)
    return median_filter(raw_time_signal, window)

In [ ]:
from sklearn.model_selection import train_test_split

all_files = sorted(dataset_root.rglob('*.data'))
if not all_files:
    raise FileNotFoundError(f'No .data files found under {dataset_root}')

def get_test_group_dir(file_path: Path) -> Path:
    # Group key: .../id_person_xx/label_xx/test_xx
    for parent in file_path.parents:
        if parent.name.startswith('test_'):
            return parent
    raise ValueError(f'Could not find test_* folder in path: {file_path}')

file_to_group = {fp: get_test_group_dir(fp) for fp in all_files}
all_groups = sorted(set(file_to_group.values()))

group_labels = [parse_path_meta(group_path)['label'] for group_path in all_groups]
train_groups, test_groups = train_test_split(
    all_groups,
    test_size=0.2,
    random_state=42,
    stratify=group_labels,
    shuffle=True,
)

train_group_set = set(train_groups)
test_group_set = set(test_groups)

train_files = [fp for fp in all_files if file_to_group[fp] in train_group_set]
test_files = [fp for fp in all_files if file_to_group[fp] in test_group_set]

# Safety check: no test_* folder is shared between train and test
shared_groups = train_group_set & test_group_set
if shared_groups:
    raise RuntimeError(f'Group leakage detected across split: {sorted(shared_groups)}')

def build_unit_cache(file_list: list[Path]) -> list[dict]:
    cache = []
    for file_path in file_list:
        df = Parser(file_path).parse()
        filtered_signal = build_unit_signal(df, median_width)
        cache.append({
            'file_path': file_path,
            'df': df,
            'filtered_signal': filtered_signal,
        })
    return cache

train_unit_cache = build_unit_cache(train_files)
test_unit_cache = build_unit_cache(test_files)

global_min = min(float(np.min(unit['filtered_signal'])) for unit in train_unit_cache)
global_max = max(float(np.max(unit['filtered_signal'])) for unit in train_unit_cache)

print(f'Total units (.data files): {len(all_files)}')
print(f'Total test_* groups: {len(all_groups)}')
print(f'Train groups: {len(train_groups)}')
print(f'Test groups: {len(test_groups)}')
print(f'Train units: {len(train_unit_cache)}')
print(f'Test units: {len(test_unit_cache)}')
print(f'Train-only global amplitude min: {global_min:.6f}')
print(f'Train-only global amplitude max: {global_max:.6f}')

In [ ]:
def build_features_df(unit_cache: list[dict], global_min: float, global_max: float) -> pd.DataFrame:
    den = global_max - global_min
    if den <= 0:
        raise ValueError('Global max equals global min; cannot perform min-max scaling')

    rows = []
    for unit in unit_cache:
        file_path = unit['file_path']
        df = unit['df']
        signal = unit['filtered_signal']

        scaled_signal = (signal - global_min) / den
        scaled_signal = np.clip(scaled_signal, 0.0, 1.0)

        mean_val = float(np.mean(scaled_signal))
        std_val = float(np.std(scaled_signal))
        median_val = float(np.median(scaled_signal))
        min_val = float(np.min(scaled_signal))
        max_val = float(np.max(scaled_signal))
        q25 = float(np.percentile(scaled_signal, 25))
        q75 = float(np.percentile(scaled_signal, 75))
        iqr = q75 - q25
        range_val = max_val - min_val
        rms = float(np.sqrt(np.mean(scaled_signal ** 2)))
        energy = float(np.sum(scaled_signal ** 2))
        zcr = float(np.mean(np.abs(np.diff(np.signbit(scaled_signal - mean_val)).astype(np.int8))))

        sk = skewness(scaled_signal)
        kt = kurtosis_excess(scaled_signal)
        sc, ss, se, dom_freq, rolloff = spectral_features(scaled_signal)

        rssi_dbm = float(df['rssi_dbm'].median())

        meta = parse_path_meta(file_path)
        rows.append({
            'file_path': str(file_path),
            **meta,
            'n_packets': int(len(df)),
            'mean': mean_val,
            'std': std_val,
            'median': median_val,
            'skew': sk,
            'kurtosis': kt,
            'spectral_centroid': sc,
            'spectral_spread': ss,
            'spectral_entropy': se,
            'dominant_freq': dom_freq,
            'spectral_rolloff_85': rolloff,
            'min': min_val,
            'max': max_val,
            'q25': q25,
            'q75': q75,
            'iqr': iqr,
            'range': range_val,
            'rms': rms,
            'energy': energy,
            'zcr': zcr,
        })

    return pd.DataFrame(rows)

train_features_df = build_features_df(train_unit_cache, global_min, global_max)
test_features_df = build_features_df(test_unit_cache, global_min, global_max)
features_df = pd.concat([train_features_df, test_features_df], ignore_index=True)

print(f'Train features shape: {train_features_df.shape}')
print(f'Test features shape: {test_features_df.shape}')
features_df.head()

In [ ]:
feature_cols = [
    'mean', 'std', 'median', 'skew', 'kurtosis',
    'spectral_centroid', 'spectral_spread', 'spectral_entropy',
    'dominant_freq', 'spectral_rolloff_85',
    'min', 'max', 'q25', 'q75', 'iqr', 'range',
    'rms', 'energy', 'zcr'
]

X_train_raw = train_features_df[feature_cols].to_numpy(dtype=np.float64)
X_test_raw = test_features_df[feature_cols].to_numpy(dtype=np.float64)

# Save feature-wise min/max from train for stable inference preprocessing
feature_min = X_train_raw.min(axis=0)
feature_max = X_train_raw.max(axis=0)

# Clip train/test features to train-derived bounds
X_train = np.clip(X_train_raw, feature_min, feature_max)
X_test = np.clip(X_test_raw, feature_min, feature_max)

scaler_pca = StandardScaler()
X_train_std = scaler_pca.fit_transform(X_train)
X_test_std = scaler_pca.transform(X_test)

pca = PCA()
X_train_pca = pca.fit_transform(X_train_std)
X_test_pca = pca.transform(X_test_std)
evr = pca.explained_variance_ratio_
cum_evr = np.cumsum(evr)

evr_df = pd.DataFrame({
    'PC': np.arange(1, len(evr) + 1),
    'EVR': evr,
    'CUM_EVR': cum_evr,
})

display(evr_df.head(10))

k_95 = int(np.searchsorted(cum_evr, 0.95) + 1)
print(f'Components for 95% variance: {k_95}')

# importance_j = sum_{k=1..K}((loading_{k,j}^2) * EVR_k), K = k_95
loadings = pca.components_[:k_95, :]
feature_importance = (loadings ** 2).T @ evr[:k_95]
importance_df = pd.DataFrame({
    'feature': feature_cols,
    'importance': feature_importance
}).sort_values('importance', ascending=False)

X_train_pca_95 = X_train_pca[:, :k_95]
X_test_pca_95 = X_test_pca[:, :k_95]

train_label_mapped = train_features_df['label'].map(label_to_class)
test_label_mapped = test_features_df['label'].map(label_to_class)

if train_label_mapped.isna().any() or test_label_mapped.isna().any():
    observed_labels = set(train_features_df['label'].unique()) | set(test_features_df['label'].unique())
    unknown_labels = sorted(observed_labels - set(label_to_class))
    raise ValueError(f'Unexpected labels found: {unknown_labels}')

y_train = train_label_mapped.to_numpy(dtype=int)
y_test = test_label_mapped.to_numpy(dtype=int)

print(f'X_train_pca_95 shape: {X_train_pca_95.shape}, y_train shape: {y_train.shape}')
print(f'X_test_pca_95 shape: {X_test_pca_95.shape}, y_test shape: {y_test.shape}')
print('Train class distribution:')
for cls in sorted(class_to_label):
    print(f'  {cls} ({class_to_label[cls]}): {(y_train == cls).sum()}')
print('Test class distribution:')
for cls in sorted(class_to_label):
    print(f'  {cls} ({class_to_label[cls]}): {(y_test == cls).sum()}')

print('Top 10 features by PCA-weighted importance (using first K components):')
display(importance_df.head(10))

In [ ]:
print(evr)

In [ ]:
plt.figure(figsize=(14, 6))
plt.bar(importance_df['feature'], importance_df['importance'])
plt.title('Feature Importance by PCA Explained Variance Contribution')
plt.xlabel('Feature')
plt.ylabel('Importance')
plt.xticks(rotation=75, ha='right')
plt.tight_layout()
plt.show()

plt.figure(figsize=(8, 4))
cum_evr = np.cumsum(evr)
plt.plot(np.arange(1, len(cum_evr) + 1), cum_evr, marker='o')
plt.title('Cumulative Explained Variance by PCA Components')
plt.xlabel('Number of components')
plt.ylabel('Cumulative explained variance')
plt.ylim(0, 1.05)
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## Multiclass Classification on PCA (95% EVR)

Folder labels are mapped to classes:
- class `0`: `label_00`
- class `1`: `label_01`
- class `2`: `label_02`
- class `3`: `label_03`

Train and tune models on PCA embeddings limited to components explaining 95% variance.

In [ ]:
import sys
import subprocess

from sklearn.model_selection import StratifiedGroupKFold, GridSearchCV
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score, classification_report
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import (
    RandomForestClassifier,
    ExtraTreesClassifier,
    HistGradientBoostingClassifier,
)
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB

# Ensure CatBoost is available in the current notebook kernel
try:
    from catboost import CatBoostClassifier
except ImportError:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'catboost', '-q'])
    from catboost import CatBoostClassifier

cv = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=42)
train_cv_groups = [str(file_to_group[fp]) for fp in train_files]

# Note: parameter spaces are intentionally broader now and may significantly increase runtime.
model_spaces = {
    'CatBoost': (
        CatBoostClassifier(
            random_seed=42,
            loss_function='MultiClass',
            verbose=False
        ),
        {
            'depth': [4, 6],
            'learning_rate': [0.03, 0.07, 0.1],
            'iterations': [200, 400],
            'l2_leaf_reg': [1, 3],
        }
    ),
    'RandomForest': (
        RandomForestClassifier(random_state=42, n_jobs=-1),
        {
            'n_estimators': [300, 600, 1000],
            'max_depth': [None, 8, 12, 24, 36],
            'min_samples_split': [2, 5, 10],
            'min_samples_leaf': [1, 2, 4],
            'max_features': ['sqrt', 'log2', None],
            'class_weight': [None, 'balanced']
        }
    ),
    'ExtraTrees': (
        ExtraTreesClassifier(random_state=42, n_jobs=-1),
        {
            'n_estimators': [300, 600],
            'max_depth': [None, 8, 12, 24],
            'min_samples_split': [2, 5, 10],
            'min_samples_leaf': [1, 2, 4],
            'max_features': ['sqrt', 'log2', None],
            'class_weight': [None, 'balanced']
        }
    ),
    'SVM': (
        SVC(probability=True, random_state=42, decision_function_shape='ovr'),
        {
            'C': [0.1, 0.5, 1.0, 5.0, 10.0],
            'kernel': ['linear', 'rbf', 'poly'],
            'gamma': ['scale', 'auto'],
            'class_weight': [None, 'balanced']
        }
    ),
    'KNN': (
        KNeighborsClassifier(),
        {
            'n_neighbors': [3, 5, 7, 11, 15],
            'weights': ['uniform', 'distance'],
            'p': [1, 2]
        }
    ),
    'GaussianNB': (
        GaussianNB(),
        {
            'var_smoothing': [1e-9, 1e-8, 1e-7, 1e-6]
        }
    ),
}

search_results = []
best_search = None
best_model_name = None

for model_name, (estimator, param_grid) in model_spaces.items():
    print(f'\nGrid search for: {model_name}')
    search = GridSearchCV(
        estimator=estimator,
        param_grid=param_grid,
        scoring='f1_macro',
        cv=cv,
        n_jobs=-1,
        refit=True
    )
    search.fit(X_train_pca_95, y_train, groups=train_cv_groups)

    best_model = search.best_estimator_
    y_pred = best_model.predict(X_test_pca_95)

    y_proba = None
    test_roc_auc_ovr = float('nan')
    if hasattr(best_model, 'predict_proba'):
        y_proba = best_model.predict_proba(X_test_pca_95)
        try:
            test_roc_auc_ovr = roc_auc_score(
                y_test,
                y_proba,
                multi_class='ovr',
                average='macro'
            )
        except ValueError:
            test_roc_auc_ovr = float('nan')

    result = {
        'model': model_name,
        'best_cv_f1_macro': search.best_score_,
        'test_f1_macro': f1_score(y_test, y_pred, average='macro'),
        'test_accuracy': accuracy_score(y_test, y_pred),
        'test_roc_auc_ovr_macro': test_roc_auc_ovr,
        'best_params': search.best_params_
    }
    search_results.append(result)

    if best_search is None or search.best_score_ > best_search.best_score_:
        best_search = search
        best_model_name = model_name

results_df = pd.DataFrame(search_results).sort_values('best_cv_f1_macro', ascending=False).reset_index(drop=True)
print('\nModel ranking by best CV F1-macro:')
display(results_df[['model', 'best_cv_f1_macro', 'test_f1_macro', 'test_accuracy', 'test_roc_auc_ovr_macro']])

best_model = best_search.best_estimator_
print(f'\nBest model: {best_model_name}')
print(f'Best CV F1-macro: {best_search.best_score_:.4f}')
print(f'Best params: {best_search.best_params_}')

best_pred = best_model.predict(X_test_pca_95)
target_names = [class_to_label[c] for c in sorted(class_to_label)]
print('\nClassification report (best model on test set):')
print(classification_report(y_test, best_pred, labels=sorted(class_to_label), target_names=target_names, digits=4))

In [ ]:
from pathlib import Path
import json
import joblib

artifacts_dir = project_root / 'artifacts' / 'multilabel_ml'
artifacts_dir.mkdir(parents=True, exist_ok=True)

model_type = 'catboost' if best_model_name == 'CatBoost' else 'sklearn'
if model_type == 'catboost':
    model_path = artifacts_dir / 'catboost_model.cbm'
    best_model.save_model(str(model_path))
else:
    model_slug = best_model_name.lower().replace(' ', '_')
    model_path = artifacts_dir / f'{model_slug}_model.joblib'
    joblib.dump(best_model, model_path)

preproc_path = artifacts_dir / 'preprocessing.joblib'
meta_path = artifacts_dir / 'metadata.json'

model_classes = sorted(class_to_label)
if hasattr(best_model, 'classes_'):
    model_classes = [int(c) for c in best_model.classes_]

# Save preprocessing artifacts required for inference
preproc_bundle = {
    'median_width': median_width,
    'global_min': float(global_min),
    'global_max': float(global_max),
    'feature_cols': feature_cols,
    'feature_min': feature_min,
    'feature_max': feature_max,
    'scaler_pca': scaler_pca,
    'pca': pca,
    'k_95': int(k_95),
    'model_type': model_type,
    'label_to_class': label_to_class,
    'class_to_label': {str(k): v for k, v in class_to_label.items()},
    'model_classes': model_classes,
}
joblib.dump(preproc_bundle, preproc_path)

meta = {
    'best_model_name': best_model_name,
    'model_type': model_type,
    'best_cv_f1_macro': float(best_search.best_score_),
    'best_params': best_search.best_params_,
    'model_path': str(model_path),
    'preprocessing_path': str(preproc_path),
    'class_mapping': {
        '0': 'label_00',
        '1': 'label_01',
        '2': 'label_02',
        '3': 'label_03',
    },
    'preprocessing_notes': {
        'feature_clip': 'Inference clips each feature to [feature_min, feature_max] from train before StandardScaler+PCA',
    },
}
meta_path.write_text(json.dumps(meta, indent=2), encoding='utf-8')

print(f'Best model name: {best_model_name}')
print(f'Model type: {model_type}')
print(f'Saved model: {model_path}')
print(f'Saved preprocessing: {preproc_path}')
print(f'Saved metadata: {meta_path}')

## Group-Level Test Evaluation (Majority Vote)

For each `test_*` group in the test split:
- run inference for all `.data` files in the group;
- choose group prediction by majority vote;
- compare with the true group label;
- compute metrics on group-level predictions.

In [ ]:
from collections import Counter
from sklearn.metrics import confusion_matrix

# Use already prepared test embedding and labels from previous cells.
file_level_pred = best_model.predict(X_test_pca_95).astype(int)
file_level_true = y_test.astype(int)

if len(test_files) != len(file_level_pred) or len(test_files) != len(file_level_true):
    raise RuntimeError("Mismatch between test_files and test arrays; rerun preprocessing cells")

records = []
for fp, y_true_i, y_pred_i in zip(test_files, file_level_true, file_level_pred):
    records.append(
        {
            "file_path": str(fp),
            "group_path": str(file_to_group[fp]),
            "true_class": int(y_true_i),
            "pred_class": int(y_pred_i),
        }
    )

file_pred_df = pd.DataFrame(records)

# Group-level majority vote over files inside each test_* group.
group_rows = []
for group_path, grp in file_pred_df.groupby("group_path", sort=True):
    pred_votes = Counter(grp["pred_class"].tolist())
    # Tie-breaker: prefer the smaller class index to keep voting deterministic.
    majority_class = sorted(pred_votes.items(), key=lambda kv: (-kv[1], kv[0]))[0][0]

    true_values = grp["true_class"].unique().tolist()
    if len(true_values) != 1:
        raise RuntimeError(
            f"Group has mixed true labels (unexpected): {group_path} -> {true_values}"
        )
    true_class = int(true_values[0])

    group_rows.append(
        {
            "group_path": group_path,
            "n_files": int(len(grp)),
            "true_class": true_class,
            "true_label": class_to_label[true_class],
            "pred_majority_class": int(majority_class),
            "pred_majority_label": class_to_label[int(majority_class)],
            "vote_counts": dict(sorted((int(k), int(v)) for k, v in pred_votes.items())),
        }
    )

group_eval_df = pd.DataFrame(group_rows).sort_values("group_path").reset_index(drop=True)
if group_eval_df.empty:
    raise RuntimeError("No groups available for group-level evaluation")

y_true_group = group_eval_df["true_class"].to_numpy(dtype=int)
y_pred_group = group_eval_df["pred_majority_class"].to_numpy(dtype=int)

print(f"Evaluated test groups: {len(group_eval_df)}")
print(f"Group-level accuracy: {accuracy_score(y_true_group, y_pred_group):.4f}")
print(f"Group-level F1-macro: {f1_score(y_true_group, y_pred_group, average='macro'):.4f}")
print("\nGroup-level classification report:")
print(
    classification_report(
        y_true_group,
        y_pred_group,
        labels=sorted(class_to_label),
        target_names=[class_to_label[c] for c in sorted(class_to_label)],
        digits=4,
        zero_division=0,
    )
)

cm = confusion_matrix(y_true_group, y_pred_group, labels=sorted(class_to_label))
cm_df = pd.DataFrame(
    cm,
    index=[f"true_{class_to_label[c]}" for c in sorted(class_to_label)],
    columns=[f"pred_{class_to_label[c]}" for c in sorted(class_to_label)],
)
print("Group-level confusion matrix:")
display(cm_df)

# Example group check from the request.
example_group = dataset_root / "id_person_01" / "label_00" / "test_01"
example_row = group_eval_df[group_eval_df["group_path"] == str(example_group)]
if not example_row.empty:
    print("\nExample group result (id_person_01/label_00/test_01):")
    display(example_row)
else:
    print("\nExample group id_person_01/label_00/test_01 was not part of the current test split.")

print("\nFirst 10 group-level predictions:")
display(group_eval_df.head(10))